# Feature Engineering (K-Means)Converted from `src/feature_engineering_kmeans.py`---

**Beschreibung:** Feature Engineering for Employee Performance ClusteringGenerates per-assignee features from issues_snapshot.csv and issues_change_history.csv

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

# ─── Project paths ───
# Safe version for Jupyter/Colab/notebooks (where __file__ usually fails)
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    # Fallback: use current working directory (most reliable in notebooks)
    BASE_DIR = Path.cwd().resolve()

RAW_DIR      = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Create processed directory if it doesn't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ─── Header ───
print("=" * 60)
print("Feature Engineering Pipeline")
print("=" * 60)

# Quick debug output (very helpful when paths are wrong)
print(f"Base directory:      {BASE_DIR}")
print(f"Raw data folder:     {RAW_DIR}")
print(f"Processed folder:    {PROCESSED_DIR}")
print(f"Processed exists?    {PROCESSED_DIR.exists()}")

Feature Engineering Pipeline
Base directory:      /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks
Raw data folder:     /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/raw
Processed folder:    /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/processed
Processed exists?    True


## Load Data

In [16]:
import pandas as pd
from pathlib import Path

# Make sure these paths are defined (from your previous setup cell)
# If not already defined, add:
# BASE_DIR = Path.cwd().resolve()
# RAW_DIR = BASE_DIR / "data" / "raw"

print("=" * 70)
print("Loading Jira Issue Snapshot & Change History")
print("=" * 70)

# ─── 1. Load main issues snapshot ───
print("\n[1/5] Loading issues_snapshot.csv...")
issues_path = RAW_DIR / "issues_snapshot.csv"

if not issues_path.exists():
    print(f"❌ File not found: {issues_path}")
    print("→ Check path, filename spelling, and current working directory")
else:
    df = pd.read_csv(issues_path, low_memory=False)
    print(f"  Loaded: {len(df):,} rows, {df.shape[1]} columns")
    print(f"  Columns: {df.columns.tolist()[:12]}{' ...' if len(df.columns) > 12 else ''}")

# ─── 2. Load change history ───
print("\n[2/5] Loading issues_change_history.csv...")
ch_path = RAW_DIR / "issues_change_history.csv"

if not ch_path.exists():
    print(f"❌ File not found: {ch_path}")
else:
    ch = pd.read_csv(ch_path, low_memory=False)
    print(f"  Loaded: {len(ch):,} rows")
    print(f"  Columns: {ch.columns.tolist()[:10]}{' ...' if len(ch.columns) > 10 else ''}")

# ─── Quick overview ───
if 'df' in locals() and 'ch' in locals():
    print("\nQuick summary:")
    print(f"  Issues snapshot: {df.shape[0]:,} tickets")
    print(f"  Change history : {ch.shape[0]:,} change records")
    if 'issue_id' in df.columns and 'issue_id' in ch.columns:
        common_ids = set(df['issue_id']) & set(ch['issue_id'])
        print(f"  Overlapping issue_ids: {len(common_ids):,} tickets")

Loading Jira Issue Snapshot & Change History

[1/5] Loading issues_snapshot.csv...
❌ File not found: /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/raw/issues_snapshot.csv
→ Check path, filename spelling, and current working directory

[2/5] Loading issues_change_history.csv...
❌ File not found: /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/raw/issues_change_history.csv


## Parse Dates

In [17]:
print("\n[2/5] Parsing dates...")

df['issue_created'] = pd.to_datetime(df['issue_created'], utc=True, errors='coerce')
df['issue_resolution_date'] = pd.to_datetime(df['issue_resolution_date'], utc=True, errors='coerce')
ch['created'] = pd.to_datetime(ch['created'], utc=True, errors='coerce')


[2/5] Parsing dates...


NameError: name 'df' is not defined

## Priority Encoding

In [18]:
print("[2/5] Encoding priorities...")

PRIORITY_MAP = {
    'Blocker': 5,
    'Highest': 4,
    'High': 3,
    'Medium': 2,
    'unknown': 2,
    'Low': 1,
    'Lowest': 0,
}

df['priority_numeric'] = df['issue_priority'].map(PRIORITY_MAP).fillna(2)

[2/5] Encoding priorities...


NameError: name 'df' is not defined

## Global Median for Fast Resolution

In [19]:
global_median_sec = df['wf_total_time'].dropna().median()

print(f"  Global median resolution time: {global_median_sec/86400:.2f} days")

NameError: name 'df' is not defined

## Reassignment Tickets

In [20]:
print("\n[3/5] Computing reassignment data from change history...")

# Tickets that were reassigned (field='assignee' appears in change history)
reassigned_issues = set(
    ch.loc[ch['field'] == 'assignee', 'issueid']
      .dropna()
      .astype(int)
)

print(f" Tickets with reassignments: {len(reassigned_issues):,}")


[3/5] Computing reassignment data from change history...


## First Status Change (First Response)

In [21]:
print("[3/5] Computing first status change times...")

# Filter only status changes and sort chronologically
status_changes = ch[ch['field'] == 'status'].copy()
status_changes = status_changes.sort_values('created')

# Get the earliest status change per issue
first_status_change = (
    status_changes.groupby('issueid')['created']
    .first()
    .reset_index()
    .rename(columns={'created': 'first_status_change'})
)

# Prepare consistent integer issue IDs for merging
df['issueid_int'] = df['id'].astype('Int64')
first_status_change['issueid'] = first_status_change['issueid'].astype('Int64')

# Merge the first status change timestamp into main dataframe
df = df.merge(
    first_status_change,
    left_on='issueid_int',
    right_on='issueid',
    how='left'
)

# Calculate time to first status change in seconds
df['first_response_sec'] = (
    df['first_status_change'] - df['issue_created']
).dt.total_seconds()

# Prevent negative values (can happen due to timezone issues or data errors)
df['first_response_sec'] = df['first_response_sec'].clip(lower=0)

[3/5] Computing first status change times...


NameError: name 'ch' is not defined

## Active Months

In [ ]:
print("[3/5] Computing active months...")df['year_month'] = df['issue_created'].dt.to_period('M')

## Per-Assignee Feature Engineering

In [22]:
import numpy as np   # ← make sure this is imported earlier or here

print("\n[4/5] Engineering per-assignee features...")

def compute_features(group):
    n = len(group)
    
    # Efficiency
    total_times = group['wf_total_time'].dropna()
    med_res_days = total_times.median() / 86400 if len(total_times) > 0 else np.nan
    avg_res_days = total_times.mean() / 86400 if len(total_times) > 0 else np.nan
    std_res_days = total_times.std() / 86400 if len(total_times) > 0 else np.nan
    pct_fast = (total_times < global_median_sec).sum() / len(total_times) if len(total_times) > 0 else np.nan
    
    # Volume
    months = group['year_month'].nunique()
    total = n
    tpm = total / months if months > 0 else 0
    
    # Complexity
    avg_prio = group['priority_numeric'].mean()
    pct_high_prio = (group['priority_numeric'] >= 3).sum() / n
    n_projects = group['issue_proj'].nunique()
    n_categories = group['issue_type'].nunique()
    
    # Quality
    pct_reopened = (group['wfe_reopened'] > 0).sum() / n
    resolution_rate = group['issue_status'].isin(['closed', 'done']).sum() / n
    avg_comments = group['issue_comments_count'].mean()
    
    # Workflow - Sole Resolver
    issue_ids = set(group['id'].dropna().astype(int))
    reassigned_in_group = len(issue_ids & reassigned_issues)
    pct_sole_resolver = 1 - (reassigned_in_group / n) if n > 0 else np.nan
    
    # First response time
    fr = group['first_response_sec'].dropna()
    avg_first_response_days = fr.mean() / 86400 if len(fr) > 0 else np.nan
    
    # Processing steps
    avg_steps = group['processing_steps'].mean()
    
    return pd.Series({
        'median_resolution_days':   med_res_days,
        'avg_resolution_days':      avg_res_days,
        'std_resolution_days':       std_res_days,
        'pct_fast_resolved':        pct_fast,
        'total_tickets':            total,
        'tickets_per_month':        tpm,
        'active_months':            months,
        'avg_priority':             avg_prio,
        'pct_high_priority':        pct_high_prio,
        'n_distinct_projects':      n_projects,
        'n_distinct_categories':    n_categories,
        'pct_reopened':             pct_reopened,
        'resolution_rate':          resolution_rate,
        'avg_comments':             avg_comments,
        'pct_sole_resolver':        pct_sole_resolver,
        'avg_first_response_days':  avg_first_response_days,
        'avg_processing_steps':     avg_steps,
    })


# ── Filter assignees ────────────────────────────────────────────────
assignee_counts = df['issue_assignee'].value_counts()
valid_assignees = assignee_counts[assignee_counts >= 5].index

df_filtered = df[df['issue_assignee'].isin(valid_assignees)].copy()

print(f"  Assignees with >= 5 tickets: {len(valid_assignees)}")
print(f"  Filtered rows: {len(df_filtered):,}")

# ── Compute features ────────────────────────────────────────────────
features_df = df_filtered.groupby('issue_assignee').apply(compute_features)
features_df = features_df.reset_index()

print(f"  Feature matrix shape: {features_df.shape}")


[4/5] Engineering per-assignee features...


NameError: name 'df' is not defined

## Handle NaN Values

In [23]:
print("\n[5/5] Handling NaN values...")

# Efficiency features: impute with median
efficiency_cols = [
    'median_resolution_days',
    'avg_resolution_days',
    'std_resolution_days',
    'pct_fast_resolved',
    'avg_first_response_days'
]

for col in efficiency_cols:
    med = features_df[col].median()
    n_nan = features_df[col].isna().sum()
    if n_nan > 0:
        print(f" Imputing {col}: {n_nan} NaN → {med:.4f}")
    features_df[col] = features_df[col].fillna(med)

# Volume / count-based features: fill missing with 0 (makes sense for counts/empty cases)
features_df['std_resolution_days'] = features_df['std_resolution_days'].fillna(0)
# Note: you had volume_cols = ['std_resolution_days'] but only used it once — can remove if not needed

# Final checks
remaining_nan = features_df.isna().sum().sum()
print(f"\n NaN remaining: {remaining_nan}")
if remaining_nan == 0:
    print("  → All NaN values handled successfully!")
else:
    print("  → Warning: some NaN values still remain")

print(f" Final feature matrix: {features_df.shape[0]} employees × {features_df.shape[1]-1} features")


[5/5] Handling NaN values...


NameError: name 'features_df' is not defined

## Save

In [24]:
from pathlib import Path   # ← make sure this is imported earlier if needed

out_path = PROCESSED_DIR / "employee_features.csv"

features_df.to_csv(out_path, index=False)

print(f"\n Saved: {out_path}")

# Quick summary
print("\nFeature summary:")
numeric_cols = features_df.select_dtypes(include='number').columns
print(features_df[numeric_cols].describe().round(3).to_string())

NameError: name 'features_df' is not defined